# 01 — Ingest, clean, enrich and export

**CareLoop Clinics — No-Show Guard**  
Run in Google Colab. Upload the three original CSV files when prompted, or keep them in the `data/` folder. This notebook creates the Part 1 hand-off files.

In [ ]:
import csv, json, time
import numpy as np
import pandas as pd
import requests
from pathlib import Path
from google.colab import files

# Upload appointments.csv, clinics.csv and followup_notes.csv
uploaded = files.upload()
DATA_DIR = Path('/content')
OUT_DIR = Path('/content/outputs')
OUT_DIR.mkdir(exist_ok=True)

appointments_path = DATA_DIR / 'appointments.csv'
clinics_path = DATA_DIR / 'clinics.csv'
notes_path = DATA_DIR / 'followup_notes.csv'


## Task 1.1 — CSV module, type conversion and rejected rows

In [ ]:
with open(appointments_path, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    headers = reader.fieldnames
    raw_rows = list(reader)

print('Headers:', headers)
print('Row count:', len(raw_rows))

def parse_row(row):
    parsed = dict(row)
    parsed['age'] = int(parsed['age'])
    parsed['fee_inr'] = int(parsed['fee_inr'])
    parsed['prior_no_shows'] = int(parsed['prior_no_shows'])
    parsed['distance_km'] = float(parsed['distance_km'])
    return parsed

accepted, rejected = [], []
for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

print('Accepted:', len(accepted), 'Rejected:', len(rejected))
with open(OUT_DIR / 'rejected_rows.json', 'w', encoding='utf-8') as f:
    json.dump(rejected, f, indent=2)


## Task 1.2 — Pandas inspection and first-pass cleaning

In [ ]:
df = pd.read_csv(appointments_path)
print('Raw shape:', df.shape)
df.info()
print('Missing values:\n', df.isnull().sum())
print('Exact duplicate rows:', df.duplicated().sum())

df = df.drop_duplicates()
print('After duplicate removal:', len(df))
df['booking_channel'] = df['booking_channel'].astype('string').str.strip().str.title()
print('After booking channel standardization:', len(df))
print('Booking channel values:', sorted(df['booking_channel'].dropna().unique().tolist()))
df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
df['booked_on'] = pd.to_datetime(df['booked_on'], errors='coerce')
print('After date conversion (before dropping):', len(df))
df = df.dropna(subset=['appt_date', 'status']).copy()
print('After missing date/status removal:', len(df))


## Task 1.3 — Validity, imputation and outlier capping

In [ ]:
for col in ['age', 'distance_km', 'fee_inr', 'prior_no_shows']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

invalid_age_count = ((df['age'] < 1) | (df['age'] > 100)).sum()
df.loc[(df['age'] < 1) | (df['age'] > 100), 'age'] = np.nan
age_median = df['age'].median()
distance_median = df['distance_km'].median()
df['age'] = df['age'].fillna(age_median).round().astype(int)
df['distance_km'] = df['distance_km'].fillna(distance_median)

cap_counts = {}
for col in ['distance_km', 'fee_inr']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    cap_counts[col] = int(((df[col] < lower) | (df[col] > upper)).sum())
    df[col] = np.clip(df[col], lower, upper)

df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)
print('Invalid ages:', invalid_age_count)
print('Age median:', age_median, 'Distance median:', distance_median)
print('Values capped:', cap_counts)
print('Retained non-null values:\n', df.count())


## Task 1.4 — Feature engineering and NumPy summary

In [ ]:
df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
df['no_show'] = (df['status'] == 'No-Show').astype(int)
feature_cols = ['age', 'distance_km', 'fee_inr', 'lead_days']
X_summary = df[feature_cols].to_numpy()
print('Column means:', X_summary.mean(axis=0))
lead_mask = df['lead_days'].to_numpy() > 14
print('Lead days > 14:', lead_mask.sum(), 'Share:', lead_mask.mean())
distance = df['distance_km'].to_numpy()
z_scores = (distance - np.mean(distance)) / np.std(distance)
print('Rows with |z| > 2:', (np.abs(z_scores) > 2).sum())
print('No-shows:', int(df['no_show'].sum()), 'of', len(df))


## Task 1.5 — Live Open-Meteo enrichment

The first hourly value is the midnight-UTC reading as specified in the brief. The code retries once and leaves `temp_c` blank if both attempts fail; it never fabricates weather values.

In [ ]:
clinics = pd.read_csv(clinics_path)
temperatures = []
for _, clinic in clinics.iterrows():
    temp = None
    last_error = None
    for attempt in range(2):
        try:
            response = requests.get(
                'https://api.open-meteo.com/v1/forecast',
                params={'latitude': clinic['latitude'], 'longitude': clinic['longitude'],
                        'hourly': 'temperature_2m', 'forecast_days': 1},
                timeout=10
            )
            if response.status_code != 200:
                raise RuntimeError(f'HTTP {response.status_code}')
            payload = response.json()
            temp = payload['hourly']['temperature_2m'][0]
            break
        except (requests.exceptions.RequestException, KeyError, IndexError, TypeError, RuntimeError) as exc:
            last_error = str(exc)
            if attempt == 0:
                time.sleep(2)
    if temp is None:
        print('Temperature unavailable:', clinic['clinic_name'], last_error)
    temperatures.append(temp)

clinics['temp_c'] = temperatures
clinics.to_csv(OUT_DIR / 'clinics_enriched.csv', index=False)
print(clinics)


## Task 1.6 — Export clean table and SQL load script

In [ ]:
final_columns = ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
                 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel',
                 'visit_type', 'reminder_sent', 'prior_no_shows', 'status',
                 'lead_days', 'no_show']
df = df.sort_values('appointment_id').copy()
print('Unique appointment IDs:', df['appointment_id'].nunique(), 'Rows:', len(df))
print('Missing values by final column:\n', df[final_columns].isna().sum())
assert df['appointment_id'].is_unique
assert df[final_columns].isna().sum().sum() == 0

clean = df[final_columns].copy()
clean['booked_on'] = clean['booked_on'].dt.strftime('%Y-%m-%d')
clean['appt_date'] = clean['appt_date'].dt.strftime('%Y-%m-%d')
clean.to_csv(OUT_DIR / 'appointments_clean.csv', index=False)

def sql_value(value):
    if pd.isna(value):
        return 'NULL'
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        return str(float(value))
    return "'" + str(value).replace("'", "''") + "'"

sql_lines = []
for _, row in clinics.iterrows():
    values = [row[c] for c in ['clinic_id','clinic_name','city','tier','latitude','longitude','temp_c']]
    sql_lines.append(f"INSERT INTO clinics VALUES ({','.join(sql_value(v) for v in values)});")
for _, row in clean.iterrows():
    sql_lines.append(f"INSERT INTO appointments VALUES ({','.join(sql_value(v) for v in row)});")
(OUT_DIR / 'load_data.sql').write_text('\n'.join(sql_lines) + '\n', encoding='utf-8')
print('Saved:', OUT_DIR / 'appointments_clean.csv')
print('SQL INSERT statements:', len(sql_lines))


## Part 1 acceptance checks

Expected with the supplied files: 1,020 raw rows; 988 type-accepted and 32 rejected in Task 1.1; 20 exact duplicates; 990 retained rows; 5 invalid ages; age median 48; distance median 8.25; 26 distance caps; 3 fee caps; 218 no-shows; 16 exported columns. Weather values depend on the live API response on the day you run the notebook.